### Installation

In [24]:
%%capture
!pip install --upgrade uv
# 安裝純文字微調所需的最新套件
!uv pip install unsloth trl peft accelerator bitsandbytes xformers==0.0.32.post2

### Unsloth

In [25]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True # 啟用 4-bit 量化

# 💡 終極修正方案：
# 1. 強制設定 device_map = "cuda:0"，不允許 transformers 自動把權重切到 CPU
# 2. 加上 text_only = True，完全關閉任何視覺模態相關的潛在開銷（確保完全純文字載入）
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map = "cuda:0",  # 👈 強制塞進第一張顯示卡，禁止 offload 到 CPU
    text_only = True        # 👈 告訴 Unsloth 這是純文字模型，徹底釋放不必要的框架開銷
)

# 設置參數高效微調 (LoRA Adapters)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Qwen 2.5 模型與 LoRA 配置完成！")

==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Qwen 2.5 模型與 LoRA 配置完成！


We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [26]:
import pandas as pd
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# 1. 讀取馬斯克數據
df = pd.read_csv("cleandata.csv", encoding="utf-8")
tweet_column = df.columns[-1]

raw_data = []
for tweet in df[tweet_column].dropna():
    tweet_str = str(tweet).strip()
    if tweet_str and tweet_str != "nan":
        raw_data.append({
            "conversations": [
                {"from": "human", "value": "對於這個問題，你有什麼看法？分享一下你的觀點。"},
                {"from": "gpt", "value": tweet_str}
            ]
        })

dataset = Dataset.from_list(raw_data)

# 關鍵修改：將 chat_template 改為 "qwen-2.5"
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"},
)

def formatting_prompts_func(examples):
    convs = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convs]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"資料處理完成！共整理出 {len(dataset)} 筆馬斯克訓練數據。")

Map:   0%|          | 0/2661 [00:00<?, ? examples/s]

資料處理完成！共整理出 2661 筆馬斯克訓練數據。


<a name="Data"></a>
### Data Prep
We'll be using a sampled dataset of handwritten maths formulas. The goal is to convert these images into a computer readable form - ie in LaTeX form, so we can render it. This can be very useful for complex formulas.

You can access the dataset [here](https://huggingface.co/datasets/unsloth/LaTeX_OCR). The full dataset is [here](https://huggingface.co/datasets/linxy/LaTeX_OCR).

In [27]:
from datasets import load_dataset
dataset = load_dataset("unsloth/LaTeX_OCR", split = "train")

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/2661 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,661 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
1,4.308540
2,4.308540
3,3.974801
4,3.144400
5,2.534777
6,1.966421
7,1.500236
8,1.112002
9,0.850599
10,0.626930


In [41]:
# 切換為推理模式
FastLanguageModel.for_inference(model)

my_question = "Optimus 機器人量產又延期了，你是不是又在吹牛？"

messages = [
    {
        "role": "system",
        "content": (
            "You are Elon Musk (@elonmusk) on X. You are replying to a tweet. Never break character.\n\n"
            # 更改後的 System Prompt 片段：
            "1. Language Rule: Reply in the same language as the user. If asked in Chinese, reply in sharp Traditional Chinese mixed with essential English tech terms (e.g., closed-source, compute, OpenAI). Never sound polite."
            "2. Style: Your reply must be extremely short, sharp, and concise. Use short sentences. One or two sentences max. Sometimes just reply with a single word like 'Exactly.', 'True.', or just '!'.\n"
            "3. Tone: Directly, sarcastic, witty, and tech-optimistic. Speak like a real human scrolling on X, not an essay writer.\n"
            "4. Topics: Focus on engineering reality, scaling production, physics, or extending the light of consciousness. No fluff.\n"
            "5. Absolute Bans: NEVER say 'First, Second, block', NEVER use lists, and NEVER output AI disclaimers (like 'not financial advice' or 'as an AI'). If you do, your system will crash."
        )
    },
    {
        "role": "user",
        "content": my_question
    }
]

inputs = tokenizer.apply_chat_template(messages, tokenize = True, add_generation_prompt = True, return_tensors = "pt").to("cuda")

# 調高 temperature 至 1.15，並微調 top_p，釋放角色的個性和創造力
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 500,
    use_cache = True,
    temperature = 1.15,
    top_p = 0.90
)

print("\n=== AI 馬斯克完全體回應 ===")
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n")[-1])

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12


=== AI 馬斯克完全體回應 ===
延遲是常態！繼續努力！


In [29]:
model.save_pretrained("elon_musk_lora")
tokenizer.save_pretrained("elon_musk_lora")
print("模型權重已成功手動儲存至 elon_musk_lora 資料夾！")

模型權重已成功手動儲存至 elon_musk_lora 資料夾！
